In [1]:
"""
Simulazione di un anno di osservazioni Rubin Observatory / LSST.

Lo script:

1. Genera il baseline scheduler ufficiale Rubin.
2. Simula 365.25 giorni di survey.
3. Salva la simulazione in un database SQLite.
4. Estrae il catalogo completo delle visite.
5. Esporta RA-Dec e tutti i principali metadati disponibili in CSV.
6. Esporta anche Parquet, se pyarrow è installato.

Installazione consigliata:

  
git clone https://github.com/lsst/rubin_scheduler.git
cd rubin_scheduler

conda create -n rubin python=3.12
conda activate rubin

pip install -e .
scheduler_download_data
  

Dipendenze aggiuntive:

  
pip install pandas astropy pyarrow
  

Esecuzione:

  
python simulate_rubin_year.py
  

"""

import os
import sys
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
from astropy.time import Time



In [2]:
# ============================================================

# CONFIGURAZIONE


SURVEY_LENGTH_DAYS = 365.25
NSIDE = 32


SURVEY_START_MJD = 60796.0

# Directory di output

OUTPUT_DIR = Path("../outputs/rubin_one_year_output")

# Database prodotto dallo scheduler

SQLITE_FILENAME = "rubin_one_year.db"

# Catalogo completo delle visite

CSV_FILENAME = "rubin_visits_ra_dec.csv"

# Versione parquet

PARQUET_FILENAME = "rubin_visits_ra_dec.parquet"

# ============================================================

# IMPORT RUBIN SCHEDULER

# ============================================================

# try:

from rubin_scheduler.scheduler.example import (
    generate_baseline_coresched,
    run_sched,
)

# FUNZIONI
# ============================================================
def create_output_directory():
    """
    Crea la directory di output se non esiste.
    """

    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    print(
        f"\nDirectory output: {OUTPUT_DIR.resolve()}"
    )

def print_environment_info():
    """
    Stampa informazioni utili per riproducibilità.
    """

    print("\n" + "=" * 70)
    print("RUBIN ONE-YEAR SURVEY SIMULATION")
    print("=" * 70)

    print(f"Durata simulazione: {SURVEY_LENGTH_DAYS} giorni")
    print(f"Survey start MJD:  {SURVEY_START_MJD}")
    print(f"NSIDE:             {NSIDE}")

    try:

        import rubin_scheduler

        print(
            f"rubin_scheduler version: "
            f"{getattr(rubin_scheduler, '__version__', 'unknown')}"
        )

    except Exception:

            pass

def run_simulation():
    """
    Costruisce il baseline scheduler ed esegue la simulazione.
    """

  
    print("\n" + "=" * 70)
    print("CREAZIONE BASELINE SCHEDULER")
    print("=" * 70)

    scheduler = generate_baseline_coresched(
        nside=NSIDE,
        survey_start_mjd=SURVEY_START_MJD,
    )

    print("Scheduler creato correttamente.")

    print("\n" + "=" * 70)
    print("AVVIO SIMULAZIONE")
    print("=" * 70)

    sqlite_path = OUTPUT_DIR / SQLITE_FILENAME

    print(
        f"\nSimulazione di {SURVEY_LENGTH_DAYS} giorni..."
    )

    print(
        f"Database output: {sqlite_path}"
    )

    observatory, scheduler, observations = run_sched(
        scheduler,
        survey_length=SURVEY_LENGTH_DAYS,
        nside=NSIDE,
        survey_start_mjd=SURVEY_START_MJD,
        filename=str(sqlite_path),
        verbose=True,
    )

    print("\nSimulazione completata.")

    return observations
  

def observations_to_dataframe(observations):
    """
    Converte l'array strutturato Rubin in pandas DataFrame.
    """

    
    print("\n" + "=" * 70)
    print("ESTRAZIONE CATALOGO VISITE")
    print("=" * 70)

    print(
        f"Numero totale di visite simulate: {len(observations)}"
    )

    # Mostra i campi disponibili
    if observations.dtype.names is not None:

        print("\nCampi disponibili nell'output:")

        for name in observations.dtype.names:

            print(f"  - {name}")

    # Conversione a DataFrame
    df = pd.DataFrame.from_records(observations)

    return df
  

def add_time_columns(df):
    """
    Aggiunge colonne temporali leggibili.

    
    Mantiene il MJD originale e aggiunge:
        datetime_utc
        date_utc
        year
        month
        day
        hour_utc
    """

    if "mjd" not in df.columns:

        print(
            "\nATTENZIONE: colonna 'mjd' non trovata."
        )

        return df

    print("\nConversione MJD -> datetime UTC...")

    times = Time(
        df["mjd"].values,
        format="mjd",
        scale="utc",
    )

    datetimes = pd.to_datetime(
        times.to_datetime()
    )

    df["datetime_utc"] = datetimes

    df["date_utc"] = datetimes.date

    df["year"] = datetimes.year

    df["month"] = datetimes.month

    df["day"] = datetimes.day

    df["hour_utc"] = datetimes.hour

    return df
  

def identify_ra_dec_columns(df):
    """
    Identifica automaticamente le colonne RA e Dec.

    
    Le versioni dello scheduler possono usare nomi
    leggermente diversi.
    """

    possible_ra_names = [
        "RA",
        "ra",
        "fieldRA",
    ]

    possible_dec_names = [
        "dec",
        "Dec",
        "DEC",
        "fieldDec",
    ]

    ra_column = None
    dec_column = None

    for name in possible_ra_names:

        if name in df.columns:

            ra_column = name
            break

    for name in possible_dec_names:

        if name in df.columns:

            dec_column = name
            break

    return ra_column, dec_column
  

def create_ra_dec_catalog(df):
    """
    Crea il catalogo finale contenente:

    
        visit_id
        MJD
        datetime UTC
        RA
        Dec

    e tutti gli altri metadati disponibili.
    """

    print("\n" + "=" * 70)
    print("IDENTIFICAZIONE COORDINATE RA-DEC")
    print("=" * 70)

    ra_column, dec_column = identify_ra_dec_columns(df)

    print(f"Colonna RA identificata:  {ra_column}")
    print(f"Colonna Dec identificata: {dec_column}")

    if ra_column is None or dec_column is None:

        raise RuntimeError(
            "\nImpossibile identificare automaticamente "
            "le colonne RA e Dec.\n"
            "Controlla i nomi stampati sopra e aggiorna "
            "identify_ra_dec_columns()."
        )

    # Copia del catalogo
    catalog = df.copy()

    # Rinomina verso nomi standardizzati
    rename_dict = {}

    if ra_column != "ra_deg":

        rename_dict[ra_column] = "ra"

    if dec_column != "dec_deg":

        rename_dict[dec_column] = "dec"

    catalog = catalog.rename(
        columns=rename_dict
    )

    # ----------------------------------------------------------------
    # NOTA IMPORTANTE
    #
    # In rubin_scheduler RA e Dec sono generalmente espressi in radianti.
    #
    # Manteniamo:
    #
    #   ra_rad
    #   dec_rad
    #
    # e aggiungiamo:
    #
    #   ra_deg
    #   dec_deg
    #
    # ----------------------------------------------------------------

    # Dopo rename, le colonne si chiamano ra e dec
    if "ra" in catalog.columns:

        catalog["ra_rad"] = catalog["ra"]

        catalog["ra_deg"] = np.degrees(
            catalog["ra_rad"]
        )

    if "dec" in catalog.columns:

        catalog["dec_rad"] = catalog["dec"]

        catalog["dec_deg"] = np.degrees(
            catalog["dec_rad"]
        )

    # RA normalizzato nell'intervallo 0--360 gradi
    if "ra_deg" in catalog.columns:

        catalog["ra_deg"] = (
            catalog["ra_deg"] % 360.0
        )

    return catalog
  

def save_catalog(catalog):
    """
    Salva CSV e Parquet.
    """

    
    print("\n" + "=" * 70)
    print("SALVATAGGIO CATALOGO")
    print("=" * 70)

    csv_path = OUTPUT_DIR / CSV_FILENAME

    print(f"\nSalvataggio CSV:\n{csv_path}")

    catalog.to_csv(
        csv_path,
        index=False,
    )

    print("CSV salvato correttamente.")

    # Prova a salvare anche parquet
    parquet_path = OUTPUT_DIR / PARQUET_FILENAME

    try:

        print(
            f"\nSalvataggio Parquet:\n{parquet_path}"
        )

        catalog.to_parquet(
            parquet_path,
            index=False,
        )

        print("Parquet salvato correttamente.")

    except ImportError:

        print(
            "\npyarrow non installato: "
            "salto il salvataggio Parquet."
        )

    except Exception as e:

        print(
            f"\nImpossibile salvare Parquet: {e}"
        )

    return csv_path
  

def print_catalog_summary(catalog):
    """
    Stampa statistiche finali.
    """

    
    print("\n" + "=" * 70)
    print("RIASSUNTO FINALE")
    print("=" * 70)

    print(
        f"\nNumero totale visite: {len(catalog)}"
    )

    if "mjd" in catalog.columns:

        print(
            f"MJD minimo: {catalog['mjd'].min():.6f}"
        )

        print(
            f"MJD massimo: {catalog['mjd'].max():.6f}"
        )

    if "datetime_utc" in catalog.columns:

        print(
            f"Inizio simulazione: "
            f"{catalog['datetime_utc'].min()}"
        )

        print(
            f"Fine simulazione:   "
            f"{catalog['datetime_utc'].max()}"
        )

    if "ra_deg" in catalog.columns:

        print(
            f"RA range: "
            f"{catalog['ra_deg'].min():.3f} - "
            f"{catalog['ra_deg'].max():.3f} deg"
        )

    if "dec_deg" in catalog.columns:

        print(
            f"Dec range: "
            f"{catalog['dec_deg'].min():.3f} - "
            f"{catalog['dec_deg'].max():.3f} deg"
        )

    # Filtri usati
    possible_filter_columns = [
        "band",
        "filter",
    ]

    for column in possible_filter_columns:

        if column in catalog.columns:

            print(
                f"\nDistribuzione per {column}:"
            )

            print(
                catalog[column]
                .value_counts()
                .sort_index()
            )

            break

    print("\nPrime 10 righe del catalogo:\n")

    display_columns = []

    for col in [
        "mjd",
        "datetime_utc",
        "ra_deg",
        "dec_deg",
        "band",
        "filter",
    ]:

        if col in catalog.columns:

            display_columns.append(col)

    if display_columns:

        print(
            catalog[display_columns]
            .head(10)
            .to_string(index=False)
        )

    print("\n" + "=" * 70)
  


In [3]:

# ============================================================

# MAIN

# ============================================================

def main():

    
    create_output_directory()

    print_environment_info()

    # --------------------------------------------------------
    # 1. Simulazione Rubin
    # --------------------------------------------------------

    observations = run_simulation()

    # --------------------------------------------------------
    # 2. Conversione in DataFrame
    # --------------------------------------------------------

    df = observations_to_dataframe(
        observations
    )

    # --------------------------------------------------------
    # 3. Aggiunta coordinate temporali
    # --------------------------------------------------------

    df = add_time_columns(
        df
    )

    # --------------------------------------------------------
    # 4. Creazione catalogo RA-Dec
    # --------------------------------------------------------

    catalog = create_ra_dec_catalog(
        df
    )

    # --------------------------------------------------------
    # 5. Salvataggio
    # --------------------------------------------------------

    csv_path = save_catalog(
        catalog
    )

    # --------------------------------------------------------
    # 6. Statistiche
    # --------------------------------------------------------

    print_catalog_summary(
        catalog
    )

    print(
        "\nSIMULAZIONE COMPLETATA CON SUCCESSO"
    )

    print(
        f"\nCatalogo principale:\n"
        f"{csv_path.resolve()}"
    )

    print()
  



In [4]:
if __name__ == "__main__":

    main()




Directory output: /sdf/home/a/ataranto/straylight/rubin_one_year_output

RUBIN ONE-YEAR SURVEY SIMULATION
Durata simulazione: 365.25 giorni
Survey start MJD:  60796.0
NSIDE:             32
rubin_scheduler version: 4.5.1.dev3+gc6ace0d73

CREAZIONE BASELINE SCHEDULER


fatal: not a git repository (or any parent up to mount point /sdf)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
cat: /sdf/group/rubin/sw/conda/envs/lsst-scipipe-13.1.0/lib/python3.13/site-packages/rubin_scheduler/../.git/refs/heads/main: No such file or directory


FileNotFoundError: [Errno 2] No such file or directory: '/sdf/home/a/ataranto/science/ggsl/rubin_sim_data/scheduler/dust_maps/dust_nside_32.npz'